# Limpieza inicial del Formulario de Contexto

- **Entrada:** `data/FORMULARIO DE CONTEXTO.csv` (la descarga original de Google Forms). **Nunca se modifica.**
- **Salida:** `data/FORMULARIO DE CONTEXTO limpio.csv`

Todas las celdas trabajan sobre `df` en memoria y solo la última guarda. Así el notebook se puede correr completo (Run All) las veces que quieras sin perder datos.

Al leer el archivo se quitan los espacios sobrantes de encabezados y respuestas (incluidos los que agrega la opción "Align columns" de algunos editores).

In [1]:
import re
import unicodedata
import pandas as pd
from pathlib import Path

# Busca la carpeta "data" subiendo desde donde se ejecuta el notebook
# (funciona igual desde la raíz del proyecto, rules/ o distributions/)
DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())

RUTA_ORIGINAL = DATA / "FORMULARIO DE CONTEXTO.csv"
RUTA_LIMPIA = DATA / "FORMULARIO DE CONTEXTO limpio.csv"

# skipinitialspace: tolera CSV "alineados" (con espacios de relleno antes de las comillas)
df = pd.read_csv(RUTA_ORIGINAL, encoding="utf-8-sig", skipinitialspace=True, dtype=str)

def limpiar_espacios(serie):
    """Quita espacios al inicio/fin y deja un solo espacio entre palabras."""
    return (serie.str.replace("\u202f", " ", regex=False)
                 .str.replace("\u00a0", " ", regex=False)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
                 .replace("", pd.NA))

df.columns = limpiar_espacios(pd.Series(df.columns)).tolist()
df = df.apply(limpiar_espacios)
df = df.drop(columns=["Marca temporal"], errors="ignore")

print("Filas:", len(df), "| Columnas:", len(df.columns))

Filas: 107 | Columnas: 41


Transformacion de Respuestas

Renombrado de los encabezados largos de la encuesta a nombres cortos.

In [2]:
# Pegar columnas que el parser partió por comas internas en el nombre
cols = list(df.columns)
broken = {
    '"En caso de haberlo considerado o decidido': '¿cuál es el principal motivo?"',
}
for inicio, fin in broken.items():
    if inicio in cols and fin in cols:
        col_completa = f"{inicio}, {fin}".strip('"')
        df[col_completa] = df[inicio].astype(str).str.strip() + ", " + df[fin].astype(str).str.strip()
        df = df.drop(columns=[inicio, fin])
        print(f"Columna reparada: {repr(col_completa)}")

nuevos_nombre = {
    # Datos generales
    "Nombre Completo del Alumno": "Nombre",
    "Sexo": "Sexo",
    "Semestre": "Semestre",
    "Escuela actual": "Escuela",
    "Numero de control o identificador unico": "ID alumno",
    "Carrera": "Carrera",
    "Grupo y Seccion": "Grupo",

    # Intención de abandono escolar
    "¿Has considerado recientemente abandonar tus estudios o has tomado ya la decisión de dejar la escuela de manera definitiva?": "Riesgo abandono",
    "En caso de haberlo considerado o decidido, ¿cuál es el principal motivo?": "Motivo abandono",

    # Infraestructura del hogar
    "¿Que tan constante diria que es la presencia de luz en casa?": "Luz",
    "¿Que tan constante diria que es la disponibilidad del agua en su casa?": "Agua",
    "¿Que tan constante diria que es la estabilidad de su internet para subir tareas pesadas en casa?": "Internet",
    "¿Cuenta con equipo de computo de uso exclusivo suyo en casa?": "Equipo propio",
    "¿Qué tan frecuente es que tu equipo de cómputo se trabe, se apague solo o sea demasiado lento para tus tareas?": "Equipo lento",
    "¿Con cuantos baños cuenta su casa?": "Banos",
    "¿Cuántos metros cuadrados (aproximados) tiene tu casa?": "Tamano casa",
    "¿Cuantas personas viven en su casa?": "Personas hogar",
    "¿Cuantas personas comparten la habitación donde duermes?": "Personas habitacion",

    # Situación económica y transporte
    "En caso de una emergencia económica de $5,000 MXN, ¿Qué tan difícil sería para ti o tu familia conseguir ese dinero en menos de 24 horas?": "Dificultad liquidez",
    "¿Cual es el promedio de ingresos por mes en tu hogar?": "Ingreso mensual",
    "¿Cuanto tiempo realiza en el transporte de su casa a la escuela de manera redonda (Ida y vuelta)?": "Tiempo transporte",
    "En general por semana, ¿Cuanto gastas por semana en transporte?": "Gasto transporte",
    "¿Qué tan seguro te sientes caminando por tu colonia o esperando el transporte público?": "Seguridad transporte",

    # Salud y limitaciones funcionales
    "En tu día a día como estudiante, ¿qué tanto limitan tus condiciones físicas o mentales tu capacidad para realizar las siguientes tareas?": "Limitaciones funcionales",
    "¿Requieres de gastos mensuales fijos adicionales para mantener tu salud y poder estudiar? (Medicamentos, terapias, prótesis, transporte especial)": "Gastos salud",
    "¿Cuentas con algún diagnóstico oficial que te permita solicitar ajustes razonables en tus materias?": "Diagnostico oficial",

    # Alimentación y consumo de sustancias
    "¿Cuantas comidas completas realizas al dia en promedio?": "Comidas diarias",
    "¿Que tan seguido inicias tus clases sin haber desayunado por falta de tiempo o dinero?": "Falta desayuno",
    "¿En el ultimo mes, tuviste que reducir porciones o saltarte una comida porque no alcanzaba el dinero?": "Inseguridad alimentaria",
    "¿Con que frecuencia consumes alcohol o tabaco?": "Consumo sustancias",
    "¿Que tan de tu presupuesto semanal se destina al consumo de sustancias (incluyendo café, tabaco, alcohol o sustancias recreativas)?": "Gasto sustancias",
    "¿Qué tan frecuente es que tu consumo de sustancias afecte tu ciclo de sueño o tu capacidad de concentración al día siguiente?": "Impacto sustancias",
    '¿Qué tanto dependes de sustancias (energizantes, café excesivo) para poder cumplir con tus entregas académicas?"': "Dependencia estimulantes",

    # Contexto familiar y apoyo social
    "¿Con quienes vives actualmente?": "Convivencia",
    "Además de estudiar, ¿Eres tú la principal persona responsable del cuidado de menores (hermanos, hijos) o de adultos dependientes (abuelos enfermos) en tu casa?": "Responsabilidad cuidados",
    "¿Eres tú la principal fuente de ingresos económicos de tu hogar?": "Proveedor principal",
    "Si este semestre reprobas una materia, ¿cuál crees que sería la reacción principal en tu casa?": "Reaccion familiar",
    "¿Qué tanto sientes que tu familia comprende la dificultad real de estudiar tu carrera?": "Comprension familiar",
    "Cuando tienes estrés por entregas o exámenes, ¿con qué frecuencia hay alguien en tu casa con quien puedas hablar y que realmente te escuche sin juzgarte?": "Apoyo emocional",
    "Alguien en tu círculo cercano (familiares o amigos de confianza) ha estudiado una carrera universitaria o ingeniería?": "Referentes universitarios",
    "Si tienes un problema administrativo en la escuela (beca, calificaciones, horarios), ¿tienes a quién acudir aparte de tus compañeros de clase para que te aconseje?": "Apoyo administrativo",
}

# Solo es error si no aparece ni el nombre original ni el nuevo
faltantes = [orig for orig, nuevo in nuevos_nombre.items()
             if orig not in df.columns and nuevo not in df.columns]
if faltantes:
    raise KeyError(f"{len(faltantes)} encabezados no coinciden: {faltantes}")

df = df.rename(columns=nuevos_nombre)
if df.columns.duplicated().any():
    raise ValueError("Hay nombres de columnas duplicados.")

print("Columnas renombradas correctamente.")

Columnas renombradas correctamente.


Limpieza de Datos: Columna "Semestre"

En este apartado se estandarizó la columna de Semestre. El principal reto fue la falta de restricciones en la encuesta, lo que generó una alta variabilidad en las respuestas de los alumnos.

Problema identificado: Heterogeneidad en los formatos de entrada.

Formatos detectados: "Segundo Semestre", "2do", "2", "Septimo", "8vo", "1er semestre".

Estrategia:
1. Se busca un número en la respuesta ("2do" → 2, "8vo" → 8).
2. Si no hay número, se busca el ordinal escrito ("Séptimo semestre" → 7), sin importar acentos ni mayúsculas.
3. Si la respuesta está vacía o no se entiende, se infiere del **Grupo** ("4 B" → 4, "Séptimo A" → 7). Estos casos quedan marcados en la columna `Semestre origen`.
4. Las respuestas en **cuatrimestres** se marcan en la columna `Periodo`, porque un 4.º cuatrimestre no equivale a un 4.º semestre.

In [3]:
ORDINALES = {
    "primer": 1, "segund": 2, "tercer": 3, "cuart": 4, "quint": 5,
    "sext": 6, "septim": 7, "setim": 7, "octav": 8, "noven": 9, "decim": 10,
}

def quitar_acentos(texto):
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))

def leer_semestre(valor):
    """Convierte '2do', 'Séptimo semestre', '8vo', '4.0', etc. a un entero 1-12."""
    if pd.isna(valor):
        return pd.NA
    texto = quitar_acentos(str(valor).lower().strip())
    numero = re.search(r"\d+", texto)
    if numero:
        n = int(numero.group())
    else:
        n = next((v for raiz, v in ORDINALES.items() if raiz in texto), None)
    return n if n is not None and 1 <= n <= 12 else pd.NA

original = df["Semestre"].copy()
desde_respuesta = original.map(leer_semestre)
desde_grupo = df["Grupo"].map(leer_semestre)

df["Semestre"] = desde_respuesta.fillna(desde_grupo).astype("Int64")
df["Semestre origen"] = "Respuesta"
df.loc[desde_respuesta.isna() & desde_grupo.notna(), "Semestre origen"] = "Grupo"
df.loc[df["Semestre"].isna(), "Semestre origen"] = pd.NA

# Algunas escuelas (Vizcaya, UAD, UTL) trabajan por cuatrimestres: se marca para no mezclarlos
df["Periodo"] = original.map(lambda v: "Cuatrimestre" if "cuatri" in quitar_acentos(str(v).lower()) else "Semestre")
print(df["Periodo"].value_counts(), "\n")

print(df["Semestre"].value_counts(dropna=False).sort_index(), "\n")

inferidos = df["Semestre origen"].eq("Grupo")
print(f"Inferidos desde Grupo: {inferidos.sum()}")
print(pd.DataFrame({"Nombre": df["Nombre"], "Respuesta original": original,
                    "Grupo": df["Grupo"], "Semestre": df["Semestre"]})[inferidos].to_string(), "\n")

vacios = df["Semestre"].isna()
print(f"Sin semestre (revisar a mano): {vacios.sum()}")
print(pd.DataFrame({"Nombre": df["Nombre"], "Respuesta original": original,
                    "Grupo": df["Grupo"]})[vacios].to_string())

Periodo
Semestre        95
Cuatrimestre    12
Name: count, dtype: int64 

Semestre
1       34
2        4
3        8
4       24
5        6
7       26
8        2
9        2
<NA>     1
Name: count, dtype: Int64 

Inferidos desde Grupo: 0
Empty DataFrame
Columns: [Nombre, Respuesta original, Grupo, Semestre]
Index: [] 

Sin semestre (revisar a mano): 1
                                 Nombre Respuesta original Grupo
53  Victoria Alejandra González Favela                   0     0


Reglas genéricas

Escalas Likert (1-5) a valores entre 0 y 1, y respuestas Sí/No a 1/0.

In [4]:
escala_directa = {1: 0.0, 2: 0.25, 3: 0.50, 4: 0.75, 5: 1.0, 6: 1.0}
escala_si_no = {"si": 1, "no": 0}

columnas_si_no = [
    "Equipo propio",             # ¿Cuenta con equipo de computo de uso exclusivo...?
    "Diagnostico oficial",       # ¿Cuentas con algún diagnóstico oficial...?
    "Inseguridad alimentaria",   # ¿En el ultimo mes, tuviste que reducir porciones...?
    "Responsabilidad cuidados",  # Además de estudiar, ¿Eres tú la principal persona responsable del cuidado...?
    "Proveedor principal",       # ¿Eres tú la principal fuente de ingresos...?
]

columnas_escala_directa = [
    "Luz",
    "Agua",
    "Internet",
    "Equipo lento",
    "Banos",
    "Seguridad transporte",
    "Limitaciones funcionales",  # En tu día a día como estudiante...
    "Reaccion familiar",         # Si este semestre reprobas una materia...
    "Comprension familiar",      # ¿Qué tanto sientes que tu familia comprende...?
    "Apoyo emocional",           # Cuando tienes estrés por entregas...
]

for columna in columnas_si_no:
    texto = df[columna].astype("string").str.strip().str.lower().map(
        lambda v: quitar_acentos(v) if isinstance(v, str) else v)
    df[columna] = texto.map(escala_si_no).astype("Int64")

for columna in columnas_escala_directa:
    df[columna] = pd.to_numeric(df[columna], errors="coerce").map(escala_directa)

# Avisar si algún valor no se pudo convertir
for columna in columnas_si_no + columnas_escala_directa:
    n = df[columna].isna().sum()
    if n:
        print(f"Aviso: {n} valores vacíos o no reconocidos en '{columna}'")

print("Reglas genéricas aplicadas.")

Reglas genéricas aplicadas.


In [5]:
df.to_csv(RUTA_LIMPIA, index=False, encoding="utf-8-sig")
print(f"Guardado en: {RUTA_LIMPIA}  ({len(df)} filas, {len(df.columns)} columnas)")

Guardado en: data/FORMULARIO DE CONTEXTO limpio.csv  (107 filas, 43 columnas)
